# All fire metrics aggregation

This notebook loads every `*_viirs_singlefire_metrics.gpkg` produced by `single_fire_metrics.ipynb`, concatenates them into one GeoDataFrame named `all_fires`, and plots intensity vs severity.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")


In [ ]:
# Directory containing files like: <FIRE_NAME>_viirs_singlefire_metrics.gpkg
OUTPUT_DIR = Path("single_fire_ouputs")

gpkg_files = sorted(OUTPUT_DIR.glob("*_viirs_singlefire_metrics.gpkg"))
if not gpkg_files:
    raise FileNotFoundError(
        f"No single-fire metric files found in: {OUTPUT_DIR.resolve()}"
    )

print(f"Found {len(gpkg_files)} files")
for p in gpkg_files:
    print(f" - {p.name}")


In [ ]:
def fire_name_from_path(path: Path) -> str:
    suffix = "_viirs_singlefire_metrics"
    stem = path.stem
    if stem.endswith(suffix):
        stem = stem[: -len(suffix)]
    return stem.replace("_", " ")


all_frames = []
for gpkg_path in gpkg_files:
    gdf = gpd.read_file(gpkg_path)
    gdf["fire_event_name"] = fire_name_from_path(gpkg_path)
    all_frames.append(gdf)

all_fires = gpd.GeoDataFrame(
    pd.concat(all_frames, ignore_index=True),
    geometry="geometry",
    crs=all_frames[0].crs,
)

print(f"All fires rows: {len(all_fires):,}")
all_fires[["fire_event_name"]].value_counts().rename("n_rows").head(20)


In [ ]:
# Save combined GeoDataFrame (optional)
all_fires_path = OUTPUT_DIR / "all_fires_metrics.gpkg"
all_fires.to_file(all_fires_path, driver="GPKG")
print(f"Saved: {all_fires_path}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

sns.scatterplot(
    data=all_fires,
    x="log_cfrp",
    y="cbi_mean",
    hue="duration_group",
    alpha=0.6,
    ax=axes[0],
)
axes[0].set_title("All fires: intensity vs severity by duration")
axes[0].set_xlabel("log_cfrp")
axes[0].set_ylabel("cbi_mean")

sns.scatterplot(
    data=all_fires,
    x="log_cfrp",
    y="cbi_mean",
    hue="fire_event_name",
    alpha=0.5,
    ax=axes[1],
)
axes[1].set_title("All fires: intensity vs severity by fire")
axes[1].set_xlabel("log_cfrp")
axes[1].set_ylabel("cbi_mean")
axes[1].legend(title="fire_event_name", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.show()
